In [1]:
# This script checks for NaN values across all results to find unreliable boosting fits before statistical analysis

import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# Define the path that will be used
DATA_ROOT = Path('/Users/zorkabozilovic/Desktop/PART1')
df = pd.read_csv(DATA_ROOT / 'all_trf_results.csv')

In [2]:
# Check which columns contain NaN values

for col in df.columns:
    n_nan = df[col].isna().sum()
    if n_nan > 0:
        print(f'{col:<25} {n_nan:>5} NaNs')

mean_r                        5 NaNs
mean_prop_explained           5 NaNs
r_trial_1                     5 NaNs
r_trial_2                     5 NaNs
r_trial_3                     5 NaNs
r_trial_4                     5 NaNs
r_trial_5                     5 NaNs
r_trial_6                     5 NaNs
r_trial_7                     5 NaNs
r_trial_8                     5 NaNs
r_trial_9                     5 NaNs
r_trial_10                    5 NaNs
r_trial_11                    5 NaNs
r_trial_12                    5 NaNs
r_trial_13                    5 NaNs
r_trial_14                    5 NaNs
r_trial_15                    5 NaNs
r_trial_16                    5 NaNs
r_trial_17                    5 NaNs
r_trial_18                    5 NaNs
r_trial_19                    5 NaNs
r_trial_20                    5 NaNs
r_trial_21                    5 NaNs
r_trial_22                    5 NaNs
r_trial_23                    5 NaNs
r_trial_24                    5 NaNs
r_trial_25                    5 NaNs
r

In [3]:
# Since all columns above show the same number of NaN values, verify that they occur in the same rows

r_cols = [f'r_trial_{i}' for i in range(1, 31)]
pe_cols = [f'pe_trial_{i}' for i in range(1, 31)]

r_nan = df[df[r_cols].isna().all(axis=1)].index
pe_nan = df[df[pe_cols].isna().all(axis=1)].index
mean_r_nan = df[df['mean_r'].isna()].index
mean_pe_nan = df[df['mean_prop_explained'].isna()].index

same_rows = (set(r_nan) == set(pe_nan) == set(mean_r_nan) == set(mean_pe_nan))
print(f'Same rows have NaN values across all columns: {same_rows}')
print(f'NaN row indices: {sorted(r_nan.tolist())}') # Get indices of rows containing NaN values

Same rows have NaN values across all columns: True
NaN row indices: [286, 322, 3314, 3886, 3922]


In [4]:
# Identify which subjects, bands, and features have NaN values (rows 286, 322, 3314, 3886, 3922 as it can be seen above)

nan_rows = df.loc[[286, 322, 3314, 3886, 3922]]

print(f'{"Subject":<10} {"Band":<10} {"Direction":<15} {"Pipeline":<20} {"Combo":<10} {"NaN r":>8} {"NaN pe":>8}')
print('-' * 90)

for _, row in nan_rows.iterrows(): # Only need the data
    n_nan_r = sum(np.isnan(row[c]) for c in r_cols) # All 30 trials have NaN values which is why mean_r also has NaN value (only use trials to double check the number)
    n_nan_pe = sum(np.isnan(row[c]) for c in pe_cols) # All 30 trials have NaN values which is why mean_pe also has NaN value (only use trials to double check the number)
    print(f'{row["subject"]:<10} {row["band"]:<10} {row["direction"]:<15} {row["pipeline"]:<20} {row["combo_name"]:<10} {n_nan_r:>8} {n_nan_pe:>8}')

Subject    Band       Direction       Pipeline             Combo         NaN r   NaN pe
------------------------------------------------------------------------------------------
Sub2       alpha      decoding        not standardized     centroid         30       30
Sub2       beta       decoding        not standardized     centroid         30       30
Sub19      alpha      encoding        not standardized     pitch            30       30
Sub2       alpha      decoding        standardized         centroid         30       30
Sub2       beta       decoding        standardized         centroid         30       30


In [5]:
# Verify NaN values are caused by boosting not learning anything (not pipeline errors) by using models from pickle files

cases = [
    ('not improved results', 'Sub2', 'alpha', 'decode_results', 'centroid'),
    ('not improved results', 'Sub2', 'beta', 'decode_results', 'centroid'),
    ('not improved results', 'Sub19', 'alpha', 'encode_results', 'pitch'),
    ('results', 'Sub2', 'alpha', 'decode_results', 'centroid'),
    ('results', 'Sub2', 'beta', 'decode_results', 'centroid'),
]

print(f'{"Pipeline":<25} {"Sub":<10} {"Band":<10} {"Feature":<10} {"h_max":>10}')
print('-' * 70)

for folder, sub, band, direction, feat in cases:
    with open(DATA_ROOT / folder / f'{sub}_{band}.pkl', 'rb') as f:
        data = pickle.load(f)
    
    h_max = float(abs(data[direction][feat]['model'].h).max())
    
    print(f'{folder:<25} {sub:<10} {band:<10} {feat:<10} {h_max:>10.6f}')

Pipeline                  Sub        Band       Feature         h_max
----------------------------------------------------------------------
not improved results      Sub2       alpha      centroid     0.000000
not improved results      Sub2       beta       centroid     0.000000
not improved results      Sub19      alpha      pitch        0.000000
results                   Sub2       alpha      centroid     0.000000
results                   Sub2       beta       centroid     0.000000


In [6]:
print('All h_max = 0 confirms boosting stopped immediately.')
print('These cases will be excluded from statistical analysis.')

All h_max = 0 confirms boosting stopped immediately.
These cases will be excluded from statistical analysis.
